# SHAP Explainability

This notebook reproduces the SHAP-based explainability analysis for XCrime-LLM. It prepares the selected event-specific data, computes SHAP values, and generates feature-importance explanations for the model predictions.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

### Configure the Explainability Environment

Install the libraries required for SHAP analysis and import the utilities used throughout the explainability workflow.

In [ ]:
# Install dependencies required by the SHAP workflow
!pip install -q "openai>=1,<2" shap

import os
import math
import json
import functools

import shap

from openai import OpenAI

### Configure SHAP Analysis

Define the model, data paths, event type, feature set, and SHAP sampling parameters used for the explainability analysis.

In [ ]:
# OpenAI client
client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

# Fine-tuned XCrime-LLM model
MODEL_NAME = os.environ["XCRIME_MODEL_NAME"]


# Dataset paths
DATA_DIR = "/content/drive/MyDrive/XCrime-LLM/data/splits"

TRAIN_CSV = f"{DATA_DIR}/master_train.csv"
TEST_CSV = f"{DATA_DIR}/master_test.csv"


# Features used by XCrime-LLM
FEATURES = [
    "last7_total",
    "last28_mean",
    "base_rate",
    "R1_influence",
    "recency",
    "dow",
    "month",
]


# Event-to-crime mapping
EVENT_TO_CRIME = {
    "EVENT_TYPE_A": "BURGLARY",
    "EVENT_TYPE_B": "ROBBERY",
    "EVENT_TYPE_C": "GRAND LARCENY",
    "EVENT_TYPE_D": "FELONY ASSAULT",
}

# Select the event type to explain
CRIME_LABEL = "EVENT_TYPE_A"
CRIME_NAME = EVENT_TO_CRIME[CRIME_LABEL]


# Anchor columns required for region-date context
DATE_COL = "date"
HAS_REGION = True
REGION_COL = "region_id"


# SHAP configuration
n_background = 15
n_explain = 50
n_samples = 32
SEED = 42

### Load and Validate SHAP Data

Load the training and test datasets, verify the required feature columns, and retain the features and metadata needed for the explainability analysis.

In [ ]:
train_df_raw = pd.read_csv(
    TRAIN_CSV,
    low_memory=False,
)

test_df_raw = pd.read_csv(
    TEST_CSV,
    low_memory=False,
)


# Required feature, crime, and anchor columns
required_cols = (
    set(FEATURES)
    | {
        "crime",
        DATE_COL,
        REGION_COL,
    }
)

missing_train = sorted(
    [
        column
        for column in required_cols
        if column not in train_df_raw.columns
    ]
)

missing_test = sorted(
    [
        column
        for column in required_cols
        if column not in test_df_raw.columns
    ]
)

if missing_train or missing_test:
    raise KeyError(
        "Missing required columns.\n"
        f"TRAIN missing: {missing_train}\n"
        f"TEST missing: {missing_test}"
    )


# Retain features and required anchor columns
need_cols = (
    list(FEATURES)
    + [
        "crime",
        DATE_COL,
        REGION_COL,
    ]
)


# Retain event_type when available for inspection/debugging
if (
    "event_type" in train_df_raw.columns
    and "event_type" in test_df_raw.columns
):
    need_cols.append(
        "event_type"
    )


# Remove any accidental duplicate column names while preserving order
need_cols = list(
    dict.fromkeys(
        need_cols
    )
)

train_df = train_df_raw[
    need_cols
].copy()

test_df = test_df_raw[
    need_cols
].copy()


print(
    f"Loaded TRAIN: {train_df.shape}, "
    f"TEST: {test_df.shape}"
)

print(
    "Columns kept:",
    need_cols,
)

### Select SHAP Background and Explanation Cohorts

Filter the training and test data to the selected crime type, then sample the background and explanation cohorts used by KernelSHAP.

In [ ]:
# Normalize crime labels for consistent filtering
train_df["crime"] = (
    train_df["crime"]
    .astype("string")
    .str.upper()
    .str.strip()
)

test_df["crime"] = (
    test_df["crime"]
    .astype("string")
    .str.upper()
    .str.strip()
)

CRIME_NAME_NORM = (
    str(CRIME_NAME)
    .upper()
    .strip()
)


# Filter TRAIN and TEST to the selected crime type
train_lbl = train_df[
    train_df["crime"] == CRIME_NAME_NORM
].copy()

test_lbl = test_df[
    test_df["crime"] == CRIME_NAME_NORM
].copy()


print(
    f"Rows for {CRIME_NAME_NORM}: "
    f"TRAIN={len(train_lbl)}, "
    f"TEST={len(test_lbl)}"
)


# Ensure the requested cohort sizes are available
if len(train_lbl) < n_background:
    raise ValueError(
        f"Not enough TRAIN rows for {CRIME_NAME_NORM}: "
        f"{len(train_lbl)} < n_background={n_background}"
    )

if len(test_lbl) < n_explain:
    raise ValueError(
        f"Not enough TEST rows for {CRIME_NAME_NORM}: "
        f"{len(test_lbl)} < n_explain={n_explain}"
    )


# Sample SHAP background and explanation cohorts
bg_df = (
    train_lbl
    .sample(
        n=n_background,
        random_state=SEED,
    )
    .reset_index(drop=True)
)

X_expl = (
    test_lbl
    .sample(
        n=n_explain,
        random_state=SEED,
    )
    .reset_index(drop=True)
)


print(
    "Background / Explain cohorts:",
    bg_df.shape,
    X_expl.shape,
)

### Construct SHAP Perturbation Prompts

Reproduce the XCrime-LLM multi-label prompt while perturbing only the selected event's features. The remaining three event rows retain their observed context for the same region-date anchor.

In [ ]:
# Fixed event order used by the fine-tuned model
EVENT_CATEGORIES = [
    "EVENT_TYPE_A",
    "EVENT_TYPE_B",
    "EVENT_TYPE_C",
    "EVENT_TYPE_D",
]

EVENT_ORDER = EVENT_CATEGORIES.copy()


# Feature configuration
USE_BASE_RATE = True
USE_R1_INFL = True
USE_M28 = True
SHOW_ANCHOR_META = True


def _make_schema_json(events):
    return (
        "{"
        + ",".join(
            f'"{event_type}":0'
            for event_type in events
        )
        + "}"
    )


def _build_system_msg(events):
    event_list = ", ".join(events)
    schema_json = _make_schema_json(events)

    content = (
        "You are a spatiotemporal analyst.\n"
        f"For the SAME (REGION, DATE), output independent 0/1 forecasts for each of: {event_list} "
        "for whether ≥1 incident will occur in the NEXT 7 DAYS.\n"
        "Rules: decide independently; use ONLY provided numeric features; no outside knowledge; "
        "output JSON ONLY (no prose).\n"
        "Feature notes: last7_total↑, last28_mean↑, base_rate↑, R1_influence↑ ⇒ higher risk; "
        "recency lower ⇒ higher risk (9999=never). dow=0–6, month=1–12.\n"
        "Return ONE compact JSON object with exactly these keys and 0/1 values:\n"
        f"{schema_json}\n"
    )

    return {
        "role": "system",
        "content": content,
    }


SYSTEM_MSG = _build_system_msg(
    EVENT_ORDER
)

SCHEMA_JSON = _make_schema_json(
    EVENT_ORDER
)


def _fmt_i(value) -> str:
    try:
        return str(int(value))
    except Exception:
        return "0"


def _fmt_f(value, nd=4) -> str:
    try:
        value = float(value)

        if np.isfinite(value):
            return f"{value:.{nd}f}"

        return f"{0.0:.{nd}f}"

    except Exception:
        return f"{0.0:.{nd}f}"


def _event_line(
    event_type: str,
    features: dict,
) -> str:

    parts = [
        f"last7_total={_fmt_i(features.get('last7_total', 0))}",
        f"recency={_fmt_i(features.get('recency', 9999))}",
    ]

    if USE_BASE_RATE:
        parts.append(
            f"base_rate={_fmt_f(features.get('base_rate', 0.0), 4)}"
        )

    if USE_R1_INFL:
        parts.append(
            f"R1_influence={_fmt_f(features.get('R1_influence', 0.0), 3)}"
        )

    if USE_M28:
        parts.append(
            f"last28_mean={_fmt_f(features.get('last28_mean', 0.0), 3)}"
        )

    return (
        f"{event_type}: "
        + " ".join(parts)
    )


def _meta_header(
    row: pd.Series,
) -> str:

    region_id = (
        int(row[REGION_COL])
        if (
            HAS_REGION
            and REGION_COL in row
        )
        else -1
    )

    anchor_date = (
        pd.to_datetime(
            row[DATE_COL],
            errors="coerce",
        )
        if DATE_COL in row
        else None
    )

    date_string = (
        str(anchor_date.date())
        if (
            anchor_date is not None
            and pd.notna(anchor_date)
        )
        else (
            str(row[DATE_COL])
            if DATE_COL in row
            else "NA"
        )
    )

    def _safe_int(
        value,
        default,
    ):
        try:
            return int(
                float(value)
            )
        except Exception:
            return default

    dow = (
        _safe_int(
            row["dow"],
            int(anchor_date.weekday()),
        )
        if (
            "dow" in row
            and pd.notna(row["dow"])
        )
        else (
            int(anchor_date.weekday())
            if (
                anchor_date is not None
                and pd.notna(anchor_date)
            )
            else -1
        )
    )

    month = (
        _safe_int(
            row["month"],
            int(anchor_date.month),
        )
        if (
            "month" in row
            and pd.notna(row["month"])
        )
        else (
            int(anchor_date.month)
            if (
                anchor_date is not None
                and pd.notna(anchor_date)
            )
            else 0
        )
    )

    # Preserve formatting used by the original SHAP implementation
    return (
        f"RID={region_id};"
        f"DT={date_string};"
        f"D={dow:02d};"
        f"M={month:02d}"
    )


def _derive_feats_raw(
    row: pd.Series,
) -> dict:

    def _g(
        value,
        default,
    ):
        try:
            if pd.isna(value):
                return default
            return value
        except Exception:
            return default

    features = {
        "last7_total": int(
            max(
                0,
                int(
                    _g(
                        row.get(
                            "last7_total",
                            0,
                        ),
                        0,
                    )
                ),
            )
        ),
        "recency": int(
            max(
                0,
                int(
                    _g(
                        row.get(
                            "recency",
                            9999,
                        ),
                        9999,
                    )
                ),
            )
        ),
    }

    if USE_BASE_RATE:
        features["base_rate"] = float(
            min(
                max(
                    float(
                        _g(
                            row.get(
                                "base_rate",
                                0.0,
                            ),
                            0.0,
                        )
                    ),
                    0.0,
                ),
                1.0,
            )
        )

    if USE_R1_INFL:
        features["R1_influence"] = float(
            max(
                0.0,
                float(
                    _g(
                        row.get(
                            "R1_influence",
                            0.0,
                        ),
                        0.0,
                    )
                ),
            )
        )

    if USE_M28:
        features["last28_mean"] = float(
            max(
                0.0,
                float(
                    _g(
                        row.get(
                            "last28_mean",
                            0.0,
                        ),
                        0.0,
                    )
                ),
            )
        )

    return features


# Crime-to-event mapping
CRIME_TO_EVENT = {
    crime_name: event_type
    for event_type, crime_name
    in EVENT_TO_CRIME.items()
}


def _make_long_lookup(
    df_raw: pd.DataFrame,
) -> pd.DataFrame:

    data = df_raw.copy()

    if (
        "R1_pressure" in data.columns
        and "R1_influence" not in data.columns
    ):
        data = data.rename(
            columns={
                "R1_pressure": "R1_influence"
            }
        )

    required = (
        {
            REGION_COL,
            DATE_COL,
            "crime",
        }
        | set(FEATURES)
    )

    missing = [
        column
        for column in required
        if column not in data.columns
    ]

    if missing:
        raise KeyError(
            f"[LOOKUP] Missing columns: {missing}"
        )

    data[DATE_COL] = (
        pd.to_datetime(
            data[DATE_COL],
            errors="coerce",
        )
        .dt.date
        .astype(str)
    )

    data["crime"] = (
        data["crime"]
        .astype("string")
        .str.upper()
        .str.strip()
    )

    data["event_type"] = (
        data["crime"]
        .map(CRIME_TO_EVENT)
    )

    data = data[
        data["event_type"].isin(
            EVENT_CATEGORIES
        )
    ].copy()

    index_cols = [
        REGION_COL,
        DATE_COL,
        "event_type",
    ]

    return data.set_index(
        index_cols
    )


# Use both TRAIN and TEST to retrieve the full event context
LOOKUP = _make_long_lookup(
    pd.concat(
        [
            train_df_raw,
            test_df_raw,
        ],
        ignore_index=True,
    )
)


def _extract_feats_from_row(
    row: pd.Series,
) -> dict:

    def _g(
        value,
        default,
    ):
        try:
            return (
                default
                if pd.isna(value)
                else value
            )
        except Exception:
            return default

    features = {
        "last7_total": int(
            max(
                0,
                int(
                    _g(
                        row.get(
                            "last7_total",
                            0,
                        ),
                        0,
                    )
                ),
            )
        ),
        "recency": int(
            max(
                0,
                int(
                    _g(
                        row.get(
                            "recency",
                            9999,
                        ),
                        9999,
                    )
                ),
            )
        ),
    }

    if USE_BASE_RATE:
        features["base_rate"] = float(
            min(
                max(
                    float(
                        _g(
                            row.get(
                                "base_rate",
                                0.0,
                            ),
                            0.0,
                        )
                    ),
                    0.0,
                ),
                1.0,
            )
        )

    if USE_R1_INFL:
        features["R1_influence"] = float(
            max(
                0.0,
                float(
                    _g(
                        row.get(
                            "R1_influence",
                            0.0,
                        ),
                        0.0,
                    )
                ),
            )
        )

    if USE_M28:
        features["last28_mean"] = float(
            max(
                0.0,
                float(
                    _g(
                        row.get(
                            "last28_mean",
                            0.0,
                        ),
                        0.0,
                    )
                ),
            )
        )

    return features


def _merge_meta_features(
    anchor_row: pd.Series,
    feature_row: pd.Series,
) -> pd.Series:

    merged = anchor_row.copy()

    for feature in FEATURES:
        merged[feature] = feature_row.get(
            feature,
            merged.get(
                feature,
                0,
            ),
        )

    return merged


def build_prompt_with_others(
    anchor_row_meta: pd.Series,
    feat_row_for_target: pd.Series,
    target_et: str,
) -> str:

    lines = []

    if SHOW_ANCHOR_META:
        lines.append(
            _meta_header(
                anchor_row_meta
            )
        )

    region_id = (
        int(
            anchor_row_meta[
                REGION_COL
            ]
        )
        if REGION_COL in anchor_row_meta
        else -1
    )

    date_key = (
        pd.to_datetime(
            anchor_row_meta[
                DATE_COL
            ],
            errors="coerce",
        )
        .date()
        .isoformat()
        if DATE_COL in anchor_row_meta
        else "NA"
    )

    target_features = (
        _derive_feats_raw(
            feat_row_for_target
        )
    )

    neutral_features = {
        "last7_total": 0,
        "recency": 9999,
    }

    if USE_BASE_RATE:
        neutral_features[
            "base_rate"
        ] = 0.0

    if USE_R1_INFL:
        neutral_features[
            "R1_influence"
        ] = 0.0

    if USE_M28:
        neutral_features[
            "last28_mean"
        ] = 0.0

    for event_type in EVENT_CATEGORIES:
        if event_type == target_et:
            event_features = (
                target_features
            )

        else:
            try:
                other_row = LOOKUP.loc[
                    (
                        region_id,
                        date_key,
                        event_type,
                    )
                ]

                event_features = (
                    _extract_feats_from_row(
                        other_row
                    )
                )

            except KeyError:
                event_features = (
                    neutral_features
                )

        lines.append(
            _event_line(
                event_type,
                event_features,
            )
        )

    return "\n".join(
        lines
    )


def one_token_tail(
    label_id: str,
    crime_name: str,
) -> str:
    """
    Diagnostic instruction used by the SHAP model wrapper.
    """
    return (
        f"(diagnostic) Target {label_id} ({crime_name}).\n"
        "Line 1: output exactly one token — 0 or 1.\n"
        "Line 2: output the compact JSON object with EXACTLY these keys "
        f"and 0/1 integers (no prose): {SCHEMA_JSON}"
    )

### Preview the SHAP Prompt

Preview the complete user prompt generated for one explanation instance before applying any SHAP perturbation.

In [ ]:
# Select one explanation instance for preview
row = X_expl.iloc[0]

# Use the instance's original feature values without perturbation
merged = _merge_meta_features(
    row,
    row,
)

# Construct the prompt for the selected target event
user_prompt = build_prompt_with_others(
    anchor_row_meta=row,
    feat_row_for_target=merged,
    target_et=CRIME_LABEL,
)

print("=== SHAP Prompt Preview ===")
print(user_prompt)

### Define the SHAP Model Wrapper

Convert the model's first-token log probabilities for the selected event into a probability of occurrence. For each SHAP perturbation, only the selected event's seven features are varied while the other three event rows retain their observed region-date context.

In [ ]:
def _call_llm(
    messages,
    model_name: str,
):
    return client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0,
        top_p=1,
        max_tokens=1,
        seed=SEED,
        logprobs=True,
        top_logprobs=5,
        stop=["\n", "{"],
    )


def p_from_toplogprobs(
    top_list,
) -> float:
    """
    Convert first-token log probabilities for 0 and 1
    into P(y=1).
    """

    token_logprobs = {}

    for token_info in top_list:
        token = (
            token_info.token or ""
        ).strip()

        if token in {"0", "1"}:
            token_logprobs[token] = float(
                token_info.logprob
            )

    # Fail if neither class appears in the returned candidates
    if (
        "0" not in token_logprobs
        and "1" not in token_logprobs
    ):
        raise RuntimeError(
            "Neither '0' nor '1' was found in top_logprobs. "
            "Check the diagnostic tail and stop settings."
        )

    # Preserve the original missing-class fallback
    if "0" not in token_logprobs:
        token_logprobs["0"] = (
            token_logprobs["1"] - 50.0
        )

    if "1" not in token_logprobs:
        token_logprobs["1"] = (
            token_logprobs["0"] - 50.0
        )

    logprob_0 = token_logprobs["0"]
    logprob_1 = token_logprobs["1"]

    # Numerically stable two-class softmax
    max_logprob = max(
        logprob_0,
        logprob_1,
    )

    log_denominator = (
        max_logprob
        + math.log(
            math.exp(
                logprob_0 - max_logprob
            )
            + math.exp(
                logprob_1 - max_logprob
            )
        )
    )

    probability_1 = math.exp(
        logprob_1 - log_denominator
    )

    # Avoid exact 0/1 for SHAP's logit link
    return float(
        np.clip(
            probability_1,
            1e-6,
            1 - 1e-6,
        )
    )


@functools.lru_cache(
    maxsize=8192
)
def _predict_p_cached(
    messages_json: str,
    model_name: str,
) -> float:

    messages = json.loads(
        messages_json
    )

    response = _call_llm(
        messages,
        model_name,
    )

    top_logprobs = (
        response
        .choices[0]
        .logprobs
        .content[0]
        .top_logprobs
    )

    return p_from_toplogprobs(
        top_logprobs
    )


def make_f_for_instance(
    anchor_row: pd.Series,
    label_id: str,
    crime_name: str,
    model_name: str,
):
    """
    Create the model function used by SHAP for one anchor.

    Only the seven features of the selected target event are
    perturbed. The remaining three event rows retain their
    observed region-date context.
    """

    # Ensure metadata fields exist for prompt construction
    for column in [
        REGION_COL,
        DATE_COL,
        "dow",
        "month",
    ]:
        if column not in anchor_row:
            anchor_row[column] = (
                anchor_row.get(
                    column,
                    np.nan,
                )
            )

    def f_instance(
        Xmatrix: np.ndarray,
    ) -> np.ndarray:

        # SHAP perturbations follow the exact FEATURES ordering
        feature_matrix = pd.DataFrame(
            Xmatrix,
            columns=FEATURES,
        )

        probabilities = []

        for _, feature_row in feature_matrix.iterrows():

            # Combine perturbed feature values with the fixed region/date anchor.
            # dow and month are FEATURES and may therefore be perturbed.
            merged_row = _merge_meta_features(
                anchor_row,
                feature_row,
            )

            # Target event uses perturbed features;
            # the other events use the observed LOOKUP context
            prompt = build_prompt_with_others(
                anchor_row_meta=merged_row,
                feat_row_for_target=merged_row,
                target_et=label_id,
            )

            messages = [
                SYSTEM_MSG,
                {
                    "role": "user",
                    "content": prompt,
                },
                {
                    "role": "user",
                    "content": one_token_tail(
                        label_id,
                        crime_name,
                    ),
                },
            ]

            messages_json = json.dumps(
                messages,
                sort_keys=True,
            )

            probability = _predict_p_cached(
                messages_json,
                model_name,
            )

            probabilities.append(
                probability
            )

        return np.array(
            probabilities,
            dtype=float,
        )

    return f_instance

### Compute Per-Instance SHAP Explanations

Compute KernelSHAP explanations for the selected event type. For each explained test instance, only the seven target-event features are perturbed while the other three event rows retain their observed region-date context.

In [ ]:
print(
    f"Explaining: {CRIME_LABEL} ({CRIME_NAME})"
)


# Collect per-instance SHAP outputs
shap_rows = []
base_vals = []
data_rows = []


# Iterate over the explanation cohort selected from TEST
for i in range(len(X_expl)):

    anchor_row = (
        X_expl
        .iloc[i]
        .copy()
    )

    # Create a per-instance model function that perturbs only
    # the selected event's FEATURES while holding the other
    # three event rows fixed to their observed context.
    f_i = make_f_for_instance(
        anchor_row,
        CRIME_LABEL,
        CRIME_NAME,
        MODEL_NAME,
    )


    # TRAIN background and the current TEST instance,
    # both restricted to the seven model features.
    bgi = bg_df[
        FEATURES
    ].values

    Xi = (
        X_expl
        .iloc[i:i + 1][FEATURES]
        .values
    )


    # Compute KernelSHAP values for this instance
    explainer_i = shap.KernelExplainer(
        f_i,
        bgi,
        link="logit",
    )

    sv_i = explainer_i.shap_values(
        Xi,
        nsamples=n_samples,
    )


    # Store SHAP values
    shap_rows.append(
        np.asarray(
            sv_i
        ).reshape(
            1,
            -1,
        )
    )


    # Normalize expected_value to a scalar
    if np.isscalar(
        explainer_i.expected_value
    ):
        base_i = float(
            explainer_i.expected_value
        )

    else:
        base_i = float(
            np.asarray(
                explainer_i.expected_value
            )
            .ravel()[0]
        )

    base_vals.append(
        base_i
    )


    # Retain the feature values for Explanation.data
    data_rows.append(
        Xi
    )


# Combine the individual explanations
shap_matrix = np.vstack(
    shap_rows
)

base_vec = np.array(
    base_vals,
    dtype=float,
)

data_matrix = np.vstack(
    data_rows
)


# Create one SHAP Explanation for the complete cohort
expl = shap.Explanation(
    values=shap_matrix,
    base_values=base_vec,
    data=data_matrix,
    feature_names=FEATURES,
)


print(
    "Done: per-instance SHAP explanations computed."
)

### Summarize and Save SHAP Results

Compute global feature importance from mean absolute SHAP values and save both the global importance scores and per-instance SHAP explanations for the selected event type.

In [ ]:
assert "expl" in globals(), (
    "Expected SHAP Explanation 'expl'."
)

assert (
    hasattr(expl, "values")
    and hasattr(expl, "base_values")
), "Malformed SHAP Explanation."

assert list(expl.feature_names) == list(FEATURES), (
    "FEATURES mismatch with expl.feature_names."
)


# Global feature importance: mean absolute SHAP value
global_imp = pd.Series(
    np.abs(expl.values).mean(axis=0),
    index=FEATURES,
    name=CRIME_NAME,
)

display(
    global_imp.sort_values(
        ascending=False
    )
)


# Output directory
OUT_DIR = (
    "/content/drive/MyDrive/"
    "XCrime-LLM/results/shap"
)

SAFE_CRIME_NAME = CRIME_NAME.replace(" ", "_")

os.makedirs(
    OUT_DIR,
    exist_ok=True,
)


# Save global importance
global_path = (
    f"{OUT_DIR}/"
    f"global_importance_{CRIME_LABEL}_{SAFE_CRIME_NAME}.csv"
)

global_imp.sort_values(
    ascending=False
).to_csv(
    global_path
)

print(
    f"[SAVE] Global importances → {global_path}"
)


# Per-instance SHAP table
phi_df = pd.DataFrame(
    expl.values,
    columns=FEATURES,
)

phi_df.insert(
    0,
    "base_value",
    expl.base_values.astype(float),
)


# Reconstruct predicted logit and probability
logit = (
    expl.base_values
    + expl.values.sum(axis=1)
)

p_hat = (
    1.0
    / (
        1.0
        + np.exp(-logit)
    )
)

phi_df["logit"] = logit
phi_df["p_hat"] = p_hat


# Include the feature values for each explained instance
x_df = pd.DataFrame(
    expl.data,
    columns=FEATURES,
).add_prefix("x_")


# Include available region/date metadata
meta_cols = []

for column in [
    REGION_COL,
    DATE_COL,
]:
    if column in X_expl.columns:
        meta_cols.append(
            column
        )


if meta_cols:
    meta_df = (
        X_expl
        .iloc[:len(phi_df)][meta_cols]
        .reset_index(drop=True)
        .copy()
    )

    out_df = pd.concat(
        [
            meta_df,
            x_df.reset_index(drop=True),
            phi_df.reset_index(drop=True),
        ],
        axis=1,
    )

else:
    out_df = pd.concat(
        [
            x_df.reset_index(drop=True),
            phi_df.reset_index(drop=True),
        ],
        axis=1,
    )


per_row_path = (
    f"{OUT_DIR}/"
    f"per_instance_phi_{CRIME_LABEL}_{SAFE_CRIME_NAME}.csv"
)

out_df.to_csv(
    per_row_path,
    index=False,
)

print(
    f"[SAVE] Per-instance SHAP table → {per_row_path}"
)

### SHAP Beeswarm Plot

Visualize the distribution and direction of SHAP values across the explained instances.

In [ ]:
shap.plots.beeswarm(
    expl,
    max_display=12,
    show=True,
)

### SHAP Feature-Importance Plot

Visualize global feature importance based on the mean absolute SHAP values.

In [ ]:
shap.plots.bar(
    expl,
    max_display=12,
    show=True,
)

### SHAP Waterfall Plot — Examples

Visualize the feature contributions for the first explained test instance.

In [ ]:
shap.plots.waterfall(
    expl[0],
    max_display=12,
    show=True,
)

In [ ]:
shap.plots.waterfall(
    expl[1],
    max_display=12,
    show=True,
)